## (1) Importing modules

In [ ]:
#!/usr/bin/python
# -*- coding: UTF-8 -*-

#__modification time__ = 2026-05-10
#__author__ = Qi Zhou, Helmholtz Centre Potsdam - GFZ German Research Centre for Geosciences
#__find me__ = qi.zhou@gfz.de, qi.zhou.geo@gmail.com, https://github.com/Qi-Zhou-Geo
# Please do not distribute this code without the author's permission

import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec

from obspy import read
from obspy.core import UTCDateTime
from obspy.clients.fdsn import Client

import scipy.signal
import scipy.signal.windows
scipy.signal.hann = scipy.signal.windows.hann

# region <add the sys.path to search for custom modules>
import sys
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parent.parent
sys.path.append(str(project_root))


print(f"Current dir: {current_dir}\n"
      f"Project root: {project_root}")
# endregion


# import the custom functions
from functions.visualize.visualize_seismic import psd_plot

## (2) Fetch some seismic data from Illgraben

#### (2-1) Let's download the online data from [9J network](https://www.fdsn.org/networks/detail/9J_2012/)

In [ ]:
print("This step may take 1 minute, please wait patiently.")

client = Client("GEOFON")

network, station, location, channel = "9J", "IGB02", "", "HHZ"
starttime = UTCDateTime("2014-07-11T22:00:00")
endtime = UTCDateTime("2014-07-13T02:00:00")

inventory = client.get_stations(network=network, station=station, location=location, channel=channel,
                                starttime=starttime, endtime=endtime, level="response")

st = client.get_waveforms(network=network, station=station, location=location, channel=channel,
                          starttime=starttime, endtime=endtime)
print(f"{UTCDateTime.now().isoformat()}: Done download data.")


# copy as a raw data
st_raw = st.copy()

st.merge(method=1, fill_value='latest', interpolation_samples=0)
st._cleanup()
st.detrend("linear")
st.detrend("demean")
st.taper(max_percentage=0.05)
st.remove_response(inventory=inventory, output="VEL", pre_filt=(0.5, 1, 50, 60), water_level=60)
st.filter("bandpass", freqmin=1.0, freqmax=50)
st.detrend("linear")
st.detrend("demean")
st.taper(max_percentage=0.05)
# we cut/trim the data from "2014-07-12T12:00:00" to "2014-07-12T20:00:00"
st.trim(starttime + 2 * 3600, endtime - 2 * 3600)

print(f"{UTCDateTime.now().isoformat()}: Done data process.")

# save to local
st.write(f"{current_dir}/data/cooked_2014-07-12to2014-07-13.mseed")

### (2-2) Or let's load the pre-downloaded data from local path

In [ ]:
st = read(f"{current_dir}/data/cooked_2014-07-12to2014-07-13.mseed")

## (3) Check the metadata and visulize the data

In [ ]:
tr = st.copy()
print(tr[0].stats)
print(f"Data type: {type(tr[0].data)}")
print(f"Data shape: {tr[0].data.shape}")

### (3-1) Waveform

In [ ]:
# note: there is a bug, when you run this code once, it will plot the figure twice.
tr.plot()

## Question 1: what's the unit of y?
## Question 2: Can you find the time when y reaches its maximum value?

In [ ]:
idx = np.argmax(np.abs(tr[0].data)) / tr[0].stats.sampling_rate
max_amp = tr[0].stats.starttime + idx
print(max_amp)

### (3-2) Power spectral density

In [ ]:
tr = st.copy()
tr.trim(UTCDateTime("2014-07-12T14:00:00"), UTCDateTime("2014-07-12T18:00:00"))

fig = plt.figure(figsize=(6, 4))
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 20])

cbar_ax = plt.subplot(gs[0])
ax = plt.subplot(gs[1])

ax, data_sps = psd_plot(fig, ax, cbar_ax, tr, fix_colorbar=True, per_lap=0.5, wlen=60, x_interval=1, max_plot_f=50)

cbar_ax.set_title("Power Spectral Density [dB]")
ax.set_xlabel("Time [UTC+0]", fontweight="bold")

plt.show()